# broadcasting-rules — ex7: batched attention scores with shape-trace debugging

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. Running the final beacon cell reports progress against the `Numpy: Vectorization and broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcasting-rules`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**The dangerous case.** When a shape *almost* matches you can get an unintended broadcast that runs silently and produces wrong values. Always shape-check (`print(x.shape, y.shape, (x*y).shape)`) when wiring up a new pipeline.

### Exercise 7 — batched attention scores with shape-trace debugging

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Build batched scaled-dot-product attention scores while printing intermediate shapes at each step.
> Keywords: attention, batched-matmul, transpose, shape-trace
> ```

**KCs targeted:** `batched-broadcast`, `axis-swap-via-transpose`, `broadcast-then-matmul`

Implement `ex7_attention_scores(Q, K)` to return the (unnormalized) batched attention score tensor of shape `(B, T, T)` from `Q` and `K`, both of shape `(B, T, D)`.

The formula is `scores = Q @ K.transpose(-2, -1) / sqrt(D)`. You must:

1. Print `Q.shape`, `K.shape`, and `K.transpose(-2, -1).shape` BEFORE doing the matmul.
2. Print the resulting `scores.shape` AFTER the matmul.
3. Apply the `1/sqrt(D)` scaling.

The `print(...)` calls are intentional — they let you *see* how the batch axis rides along while the inner matmul reshape happens on the last two axes only. Comment them out later if you want; the test ignores stdout.

In [ ]:
def ex7_attention_scores(Q: Tensor, K: Tensor) -> Tensor:
    import math
    D = Q.shape[-1]
    K_T = K.transpose(-2, -1)
    print(f'Q.shape       = {tuple(Q.shape)}')
    print(f'K.shape       = {tuple(K.shape)}')
    print(f'K_T.shape     = {tuple(K_T.shape)}')
    scores = Q @ K_T
    print(f'scores.shape  = {tuple(scores.shape)}  (before scaling)')
    return scores / math.sqrt(D)


<details><summary>Solution</summary>

```python
def ex7_attention_scores(Q: Tensor, K: Tensor) -> Tensor:
    import math
    D = Q.shape[-1]
    K_T = K.transpose(-2, -1)
    print(f'Q.shape       = {tuple(Q.shape)}')
    print(f'K.shape       = {tuple(K.shape)}')
    print(f'K_T.shape     = {tuple(K_T.shape)}')
    scores = Q @ K_T
    print(f'scores.shape  = {tuple(scores.shape)}  (before scaling)')
    return scores / math.sqrt(D)
```

**What broadcast is doing here.** `Q @ K.transpose(-2, -1)` is a *batched* matmul: PyTorch broadcasts the leading `(B,)` axis automatically and runs the matmul on the trailing `(T, D) @ (D, T) → (T, T)`. If `Q` were `(T, D)` and `K` were `(B, T, D)`, broadcasting would still work — `Q` would be promoted to `(1, T, D)` and replicated across the batch. That's how 'shared query, batched keys' lookup tables work.

**Why print the shapes.** When `Q` and `K` come from different upstream code paths it's easy to feed in `(T, B, D)` by accident — the matmul will still run, but you get attention scores between batch slots instead of sequence positions. The print-then-look loop catches that in one cycle.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()